# 03 · Fase 3 · Visualizar y contar

**Objetivo (45 min):** 3–4 gráficos que se entiendan **sin haber visto el código** y una historia de 2 minutos.

> **No gana el gráfico más bonito.** Gana el que se entiende a la primera y dice algo verdadero.

**Ruta**

1. Qué gráfico para qué pregunta
2. Un gráfico que miente vs. uno honesto
3. Una función para hacer gráficos "de entrega"
4. Del gráfico al mensaje: titulares
5. Estructura de las diapositivas y del pitch

In [ ]:
# ⚙️ CELDA DE PREPARACIÓN · ejecútala siempre la primera (Shift + Enter)
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 30)   # que se vean todas las columnas
pd.set_option("display.width", 200)

URL_BASE = "https://innosoft-igualdad-sostenibilidad.vercel.app/data/csv/"


def ruta_de(nombre_fichero):
    """Devuelve dónde está un CSV del datatón.

    1. Si está en la carpeta datos/csv del repositorio, usa esa copia (sin internet).
    2. Si no (por ejemplo, en Google Colab), usa la URL de la web del datatón.
    """
    for carpeta in [Path("datos/csv"), Path("../datos/csv")]:   # rutas RELATIVAS
        if (carpeta / nombre_fichero).exists():
            return carpeta / nombre_fichero
    return URL_BASE + nombre_fichero


def leer(nombre_fichero):
    """Lee un CSV del datatón con las opciones que funcionan para los 15 ficheros."""
    return pd.read_csv(
        ruta_de(nombre_fichero),
        sep=";",               # las columnas se separan con punto y coma
        decimal=",",           # 27,5 significa 27.5
        thousands=".",         # 1.835.875 significa 1835875
        encoding="utf-8-sig",  # quita un carácter invisible (BOM) del principio
        na_values=["."],       # el INE escribe "." cuando no hay dato
    )


print("✅ Preparación lista")

## 1. Qué gráfico para qué pregunta

| Quiero mostrar… | Gráfico | Ejemplo |
|---|---|---|
| Comparar categorías | **Barras** (horizontales si las etiquetas son largas) | Brecha por provincia |
| Evolución u orden natural | **Líneas** | Temporalidad por grupo de edad |
| Dos grupos en varias categorías | **Barras agrupadas** o **puntos unidos** (*dumbbell*) | Hombres vs. mujeres por ocupación |
| Distribución | **Histograma** o **caja (boxplot)** | Salarios de los microdatos |
| Parte de un todo | **Barra apilada al 100 %** (evita las tartas con muchas porciones) | % de mujeres por ocupación |

## 2. Un gráfico que miente vs. uno honesto

In [ ]:
s = leer("DS-005_salario_sexo_andalucia_2024.csv").set_index("sexo")
valores = s.loc[["Hombres", "Mujeres"], "salario_eq_tiempo_completo"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))

# ❌ Eje Y que no empieza en 0: una diferencia del 8 % parece del 80 %
ax1.bar(valores.index, valores.values, color=["red", "green"])
ax1.set_ylim(24500, 27300)
ax1.set_title("MAL: ¿brecha gigante?")

# ✅ Eje desde 0, título-mensaje, unidades, fuente y etiquetas con el valor
ax2.bar(valores.index, valores.values, color=["#3b6fb6", "#d9822b"])
for i, v in enumerate(valores.values):
    ax2.text(i, v + 300, f"{v:,.0f} €".replace(",", "."), ha="center")
ax2.set_ylim(0, 30000)
ax2.set_ylabel("€ al año (equivalente a tiempo completo)")
ax2.set_title("BIEN: con la misma jornada, ellas cobran un 8,2 % menos", fontsize=10)
fig.text(0.01, 0.01, "Fuente: IECA · MCVL 2024 (DS-005). Andalucía.", fontsize=8, color="gray")

fig.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

**Checklist de un gráfico de entrega**

- [ ] **Título que es una conclusión**, no una descripción ("La brecha se abre a partir de los 30" > "Brecha por edad").
- [ ] **Ejes con nombre y unidades** (€, %, miles de personas…).
- [ ] Barras **desde 0**.
- [ ] **Fuente**, año y territorio en pequeño.
- [ ] Colores con significado (y que se distingan en blanco y negro / daltonismo: azul–naranja funciona bien).
- [ ] Si son datos sintéticos, **dilo en el propio gráfico**.

## 3. Una función para no repetir

In [ ]:
COLORES = {"Hombres": "#3b6fb6", "Mujeres": "#d9822b"}
Path("figuras").mkdir(exist_ok=True)


def preparar_grafico(titulo, fuente, xlabel="", ylabel="", figsize=(8, 4.5)):
    """Crea una figura con título-titular a la izquierda y la fuente abajo."""
    fig, ax = plt.subplots(figsize=figsize)
    fig.suptitle(titulo, x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.text(0.01, 0.01, fuente, fontsize=8, color="gray")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)
    return fig, ax


def guardar(fig, nombre):
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(f"figuras/{nombre}.png", dpi=150)
    plt.show()

### Ejemplo A · Líneas: temporalidad por edad y sexo

In [ ]:
temp = leer("DS-008_temporalidad_edad_sexo_andalucia_2025.csv")
t = temp.query("sector == 'Total' and grupo_edad != 'Total' and sexo != 'Ambos sexos'")
tabla = t.pivot(index="grupo_edad", columns="sexo", values="tasa_temporalidad")
tabla = tabla.reindex(["16-19", "20-24", "25-29", "30-34", "35-39", "40-44", "45-54", "55-64", "65 y más"])

fig, ax = preparar_grafico("La mitad de los asalariados de 20–24 años tiene contrato temporal",
                           "Fuente: IECA · EPA 2025 (DS-008). Andalucía, total sectores.",
                           "Grupo de edad", "Tasa de temporalidad (%)")
for sexo in ["Hombres", "Mujeres"]:
    ax.plot(tabla.index, tabla[sexo], marker="o", label=sexo, color=COLORES[sexo])
ax.set_ylim(0, 100); ax.legend(frameon=False)
guardar(fig, "A_temporalidad_edad")

### Ejemplo B · Barras horizontales ordenadas: diferencia por provincia

In [ ]:
prov = leer("DS-015_temporalidad_provincia_sexo_andalucia_2025.csv")
tp = prov.pivot(index="provincia", columns="sexo", values="tasa_temporalidad")
tp["dif"] = tp["Mujeres"] - tp["Hombres"]
tp = tp.drop("Andalucía").sort_values("dif")

fig, ax = preparar_grafico("En 7 de 8 provincias, la temporalidad es mayor entre las mujeres",
                           "Fuente: IECA · EPA 2025 (DS-015). Diferencia en puntos porcentuales (pp).",
                           "Temporalidad mujeres − hombres (pp)")
colores = ["#d9822b" if d > 0 else "#3b6fb6" for d in tp["dif"]]
ax.barh(tp.index, tp["dif"], color=colores)
ax.axvline(0, color="black", linewidth=0.8)
for y, d in enumerate(tp["dif"]):
    ax.text(d + (0.15 if d > 0 else -0.15), y, f"{d:+.1f}", va="center", ha="left" if d > 0 else "right")
guardar(fig, "B_temporalidad_provincia")

### Ejemplo C · Puntos unidos (*dumbbell*): hombres vs. mujeres por ocupación

In [ ]:
micro = leer("DS-013_microdatos_sinteticos_tic.csv")
pm = micro.pivot_table(index="ocupacion", columns="sexo", values="salario_hora", aggfunc="median").sort_values("Hombre")

fig, ax = preparar_grafico("En todas las ocupaciones, la mediana por hora de ellas es menor",
                           "Datos SINTÉTICOS (DS-013): ilustran el método, no describen personas reales.",
                           "Salario mediano por hora (€)")
ax.hlines(pm.index, pm["Mujer"], pm["Hombre"], color="lightgray", linewidth=3)
ax.scatter(pm["Hombre"], pm.index, color=COLORES["Hombres"], label="Hombres", zorder=3)
ax.scatter(pm["Mujer"], pm.index, color=COLORES["Mujeres"], label="Mujeres", zorder=3)
ax.legend(frameon=False, loc="lower right")
guardar(fig, "C_ocupacion_dumbbell")

### Ejemplo D · Barra apilada al 100 %: ¿quién ocupa cada puesto?

In [ ]:
reparto = pd.crosstab(micro["ocupacion"], micro["sexo"], normalize="index") * 100
reparto = reparto.sort_values("Mujer")

fig, ax = preparar_grafico("Las mujeres son minoría en las ocupaciones mejor pagadas",
                           "Datos SINTÉTICOS (DS-013).", "% de personas en la ocupación")
ax.barh(reparto.index, reparto["Mujer"], color=COLORES["Mujeres"], label="Mujeres")
ax.barh(reparto.index, reparto["Hombre"], left=reparto["Mujer"], color=COLORES["Hombres"], label="Hombres")
ax.axvline(50, color="white", linestyle="--", linewidth=1)
ax.set_xlim(0, 100); ax.legend(frameon=False, loc="lower right", ncols=2)
guardar(fig, "D_reparto_ocupacion")

## 4. Del gráfico al mensaje

Cada gráfico responde a **una** pregunta y tiene **un** titular. Prueba esta fórmula:

> **[Qué pasa] + [a quién] + [cuánto]** → "La temporalidad de las mujeres de 25–29 años supera en 8,5 puntos a la de los hombres."

Y la regla de oro del datatón: **si no puedes explicar tu conclusión en una frase, no la entregues.** El jurado preguntará: *"¿y esto qué implica?"*.

## 5. Las diapositivas (3–4) y el pitch (2 minutos)

| Diapositiva | Contenido | Tiempo del pitch |
|---|---|---|
| **1 · La pregunta** | Nombre del equipo, la pregunta en grande, por qué importa | 20 s |
| **2 · Método y datos** | Qué datasets (fuente, año, territorio), qué definición de brecha, qué limpiaste | 20 s |
| **3 · Hallazgos** | 2–3 gráficos con titular | 50 s |
| **4 · Conclusión y acción** | Una frase de conclusión, **qué NO se puede concluir**, y **una acción concreta** (empresa, administración o universidad) | 30 s |

**Ejemplos de acciones concretas** (la rúbrica las valora):

- *Universidad:* publicar anualmente el salario de inserción de sus egresados por sexo y titulación (hoy es una brecha de datos).
- *Empresa TIC:* auditoría salarial por ocupación y jornada, no solo de la media global.
- *Administración:* que el IECA publique el cruce sector TIC × sexo × edad × contrato.

Guarda las diapositivas como **PDF** con el nombre del equipo. Los PNG de la carpeta `figuras/` se pueden arrastrar directamente a Google Slides, PowerPoint o Canva.

In [ ]:
# Tu turno 👇